# B2-022 — Session 4: Reparameterization and the ELBO

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

A variational autoencoder (VAE) is an autoencoder whose code is random and whose code space carries a prior, so that decoding prior samples produces new data. A trained VAE supports two downstream operations: sampling new data from the prior, and straight-line interpolation between the posterior means of two inputs (both are carried out in Session 5).
This session builds the probabilistic model behind it, derives its training objective (the evidence lower bound, ELBO) from Session 2's Jensen inequality, explains why naive sampling blocks gradients, and introduces the reparameterization $z=\mu+\sigma\odot\varepsilon$ with a log-variance head.
It ends with the exact per-example negative-ELBO bookkeeping that p12, p19, and p23 use.

## 1. Latent-variable models $p(x\mid z)\,p(z)$

A **latent-variable model** explains each observation $x$ by an unobserved cause $z$:

1. draw a latent $z$ from a **prior** $p(z)$;
2. draw the observation from a **likelihood** (decoder) $p(x\mid z)$.

The joint is $p(x,z)=p(x\mid z)\,p(z)$ (Book 1 `conditional-probability`). The model's probability of an observation, the **evidence** or **marginal likelihood**, sums or integrates out the latent: $p(x)=\sum_z p(x\mid z)p(z)$ for discrete $z$, or $p(x)=E_{z\sim p(z)}[p(x\mid z)]$ in general.
The **posterior** $p(z\mid x)=p(x\mid z)p(z)/p(x)$ is Book 1 `bayes-rule`: it says which latents plausibly produced this $x$.

**Discrete toy model** (used throughout this session). Let $z\in\{0,1\}$ with $p(z=0)=p(z=1)=0.5$. For one observed $x$, suppose $p(x\mid z=0)=0.2$ and $p(x\mid z=1)=0.6$. Then
$$p(x)=0.5\cdot0.2+0.5\cdot0.6=0.4,\qquad p(z\mid x)=\Big(\frac{0.1}{0.4},\frac{0.3}{0.4}\Big)=(0.25,0.75).$$

**Why this gets hard.** In a VAE, $z\in\mathbb R^2$ (or more) is continuous and $p(x\mid z)$ is a neural network, so $p(x)=\int p(x\mid z)p(z)\,dz$ has no closed form, and neither does the posterior. Maximum likelihood, which needs $\log p(x)$, is therefore not directly available. The rest of the session works around that.

In this unit the prior is always the standard normal $p(z)=N(0,I_k)$.

**Checkpoint 1A.** In the toy model, change $p(x\mid z=1)$ to $0.2$. What are $p(x)$ and $p(z\mid x)$?

**Checkpoint 1B.** Why can't a VAE simply maximize $\log p(x)$ with autograd?

**Collected answers.** $p(x)=0.2$ and $p(z\mid x)=(0.5,0.5)$: the observation is equally likely under both causes, so the posterior equals the prior. Because $p(x)$ is an integral of a neural network over all of $\mathbb R^k$ with no closed form, so neither its value nor its gradient can be computed exactly.

## 2. The Gaussian decoder and the reconstruction term

A VAE decoder network maps a latent $z$ to a vector $\hat x=g_\theta(z)$ of shape `(d,)`. To make it a probability model, interpret $\hat x$ as the mean of a Gaussian with identity covariance: $p(x\mid z)=N(x;\hat x,I_d)$.
By Session 1's log-density with $\Sigma=I$ (so $\log\det\Sigma=0$ and the quadratic form is a squared length),
$$-\log p(x\mid z)=\tfrac12\lVert x-\hat x\rVert^2+\tfrac d2\log2\pi=\tfrac12\sum_{j=1}^d(x_j-\hat x_j)^2+\text{const}.$$
So the **reconstruction term** for one example is **half the sum of squared errors over its features**. The constant $\frac d2\log2\pi$ does not depend on any parameter, so this unit drops it, and says so.

Two consequences matter for the bookkeeping in Section 7.
First, the sum is over **features**, not a mean: an $8$-feature example contributes eight squared errors, and the KL term (Section 3) is measured on the same per-example scale. Replacing the sum with a mean silently divides the reconstruction term by $d$ and changes the balance between the two terms (Session 5 shows what that does).
Second, a decoder with a different variance $s^2$ would give $\frac{1}{2s^2}\sum_j(x_j-\hat x_j)^2$ plus a constant: the identity-covariance choice is a modelling decision, pinned here so every practice agrees.

**Checkpoint 2A.** For $x=(1,2)$ and $\hat x=(0.5,2)$, give the reconstruction term with the constant dropped.

**Checkpoint 2B.** What multiplies the squared-error sum if the decoder variance is $s^2=0.5$ instead of $1$?

**Collected answers.** $\frac12(0.25+0)=0.125$. $\frac{1}{2s^2}=1$: the squared errors are weighted twice as heavily as with unit variance.

## 3. The approximate posterior $q(z\mid x)$

Since the true posterior $p(z\mid x)$ is intractable, a VAE trains a second network, the **encoder**, to output an approximation
$$q_\phi(z\mid x)=N\big(\mu_\phi(x),\operatorname{diag}(\sigma^2_\phi(x))\big).$$
For each input row the encoder produces two vectors of length $k$: a mean `mu` and a log-variance `logvar` $=\log\sigma^2$ (Session 2, Section 6). In code, a shared hidden layer feeds two separate linear heads:

```python
h = F.relu(self.hidden(x))          # (B, hidden)
mu = self.mu_head(h)                # (B, k), any real values
logvar = self.logvar_head(h)        # (B, k), any real values; variance = logvar.exp() > 0
```

Using one network for every $x$ (rather than separate parameters per data point) is called **amortized inference**: the encoder learns a rule that maps any input, including held-out inputs, to its approximate posterior.

**The KL to the prior** is now available in closed form, per example, summed over latent coordinates:
$$\mathrm{KL}\big(q_\phi(z\mid x)\,\|\,p(z)\big)=\tfrac12\sum_{j=1}^k\big(\mu_j^2+e^{\ell_j}-\ell_j-1\big),\quad \ell=\texttt{logvar}.$$
Its direction is the reverse KL of Session 2 (model distribution first), which is why VAE posteriors tend to be compact rather than spread over every plausible explanation.

**Checkpoint 3A.** What shapes do `mu` and `logvar` have for a batch of $B=16$ rows and $k=3$ latents, and what shape is the per-example KL?

**Checkpoint 3B.** Why does the encoder output `logvar` rather than $\sigma^2$ directly?

**Collected answers.** `mu` and `logvar` are `(16,3)`; the per-example KL is `(16,)` after summing over the latent axis. A linear head outputs any real number, and `logvar.exp()` turns every real number into a positive variance; a raw variance head could go negative.

## 4. The ELBO from Jensen's inequality

**Derivation (worked).** Fix an observation $x$ and any density $q(z\mid x)$ that is positive wherever $p(x,z)$ is. Multiply and divide by $q$ inside the evidence:
$$\log p(x)=\log\int p(x,z)\,dz=\log\int q(z\mid x)\frac{p(x,z)}{q(z\mid x)}\,dz=\log E_{z\sim q}\Big[\frac{p(x,z)}{q(z\mid x)}\Big].$$
Apply Jensen's inequality for $\log$ (Session 2, expectation form, with $Y=p(x,z)/q(z\mid x)>0$):
$$\log p(x)\ \ge\ E_{z\sim q}\Big[\log\frac{p(x,z)}{q(z\mid x)}\Big]=:\mathrm{ELBO}(x).$$
Split $\log p(x,z)=\log p(x\mid z)+\log p(z)$:
$$\mathrm{ELBO}(x)=E_{z\sim q}[\log p(x\mid z)]-E_{z\sim q}\Big[\log\frac{q(z\mid x)}{p(z)}\Big]=\underbrace{E_{z\sim q}[\log p(x\mid z)]}_{\text{reconstruction}}-\underbrace{\mathrm{KL}\big(q(z\mid x)\,\|\,p(z)\big)}_{\text{regularizer}}.$$
So the ELBO is a **lower bound on the log-evidence** that can be computed: the KL is closed-form (Section 3) and the reconstruction expectation can be estimated by sampling $z$.

**The gap is a KL.** The exact identity is $\log p(x)=\mathrm{ELBO}(x)+\mathrm{KL}\big(q(z\mid x)\,\|\,p(z\mid x)\big)$, so maximizing the ELBO over $q$ simultaneously tightens the bound and pulls $q$ toward the true posterior. Deriving this identity from Bayes' rule is p16(a).

**Numerical check with the discrete toy model.** With $q=(0.3,0.7)$ and joint values $p(x,z)=(0.1,0.3)$:
$\mathrm{ELBO}=0.3\log\frac{0.1}{0.3}+0.7\log\frac{0.3}{0.7}=-0.3295837-0.5931085=-0.9226922$, while $\log p(x)=\log0.4=-0.9162907$. The gap is $0.0064015$, and indeed $\mathrm{KL}(q\|p(z\mid x))=0.3\log\frac{0.3}{0.25}+0.7\log\frac{0.7}{0.75}=0.0546965-0.0482950=0.0064015$. With $q$ equal to the true posterior $(0.25,0.75)$, the gap is $0$.

**Checkpoint 4A.** In the toy model, what is the ELBO when $q=(0.25,0.75)$?

**Checkpoint 4B.** Which step of the derivation needs $q(z\mid x)>0$ wherever $p(x,z)>0$?

**Collected answers.** $\log p(x)=-0.916291$, because $q$ equals the true posterior and the gap $\mathrm{KL}(q\|p(z\mid x))$ is $0$. The multiply-and-divide step: dividing by $q$ where it is $0$ would drop part of the integral (and the ratio would be undefined).

In [ ]:
import math

prior = (0.5, 0.5)
likelihood = (0.2, 0.6)                                   # p(x | z) for z = 0, 1
joint = tuple(p * l for p, l in zip(prior, likelihood))   # p(x, z)
evidence = sum(joint)
posterior = tuple(j / evidence for j in joint)
for q in [(0.3, 0.7), (0.25, 0.75), (0.9, 0.1)]:
    elbo = sum(qi * math.log(ji / qi) for qi, ji in zip(q, joint))
    gap = sum(qi * math.log(qi / pi) for qi, pi in zip(q, posterior))
    print(f"q={q}: ELBO={elbo:.6f}  log p(x)={math.log(evidence):.6f}  KL(q||posterior)={gap:.6f}")
    assert math.isclose(elbo + gap, math.log(evidence), abs_tol=1e-12, rel_tol=0.0)

## 5. Why sampling blocks gradients

Training needs the gradient of the ELBO's reconstruction term $E_{z\sim q_\phi}[\log p_\theta(x\mid z)]$ with respect to the **encoder** parameters $\phi$.
The simplest estimate draws one $z\sim q_\phi(z\mid x)$ and evaluates $\log p_\theta(x\mid z)$.
But a draw from a sampler is just a number handed back by a random-number routine: there is no formula inside the computation graph that says how the drawn value would change if $\mu$ or $\sigma$ changed.
Autograd therefore has no path from the loss back through $z$ to $\mu$ and `logvar`, and the encoder receives **no reconstruction gradient** (only the KL term, which is closed-form, still reaches it).

PyTorch makes this explicit. `torch.distributions.Normal(mu, std).sample()` runs without gradient tracking: its result has `requires_grad == False` and no `grad_fn`, even when `mu` requires gradients.
Other ways to break the path are `z = z.detach()`, building `z` from `mu.item()`, or drawing `torch.normal(...)` from the detached values.
A pipeline with this defect can still train its decoder and its KL, so the loss decreases and the bug is easy to miss: the encoder's means simply stop learning anything about reconstruction.
The diagnosis is a **gradient-flow probe**: build the loss on one batch, call `backward()`, and check that `mu`'s gradient (or the encoder head's weight gradient) is not `None` and not identically zero for the reconstruction term alone.

**Checkpoint 5A.** After `z = Normal(mu, std).sample()` and `loss = ((x - decoder(z))**2).sum()`, what is `mu.grad` after `loss.backward()`?

**Checkpoint 5B.** Why can a VAE with this defect still show a decreasing training loss?

**Collected answers.** `None`: no graph connects `z` to `mu` (the reconstruction loss also never touches `mu` another way). The decoder still learns to reconstruct from whatever codes it gets, and the KL term still shrinks, so the total falls even though the encoder ignores reconstruction.

## 6. Reparameterization $z=\mu+\sigma\odot\varepsilon$ and the log-variance head

**The trick.** Draw the randomness first, from a fixed distribution that does not depend on any parameter: $\varepsilon\sim N(0,I_k)$. Then build the latent as a deterministic, differentiable function of the encoder outputs:
$$z=\mu+\sigma\odot\varepsilon,\qquad \sigma=\exp(\tfrac12\,\texttt{logvar}).$$
By Session 1's rule $\operatorname{Cov}(Az+b)=A\operatorname{Cov}(z)A^\top$ with $A=\operatorname{diag}(\sigma)$ and $b=\mu$, $z$ has mean $\mu$ and covariance $\operatorname{diag}(\sigma^2)$: exactly $q(z\mid x)$. Nothing about the distribution changed; only where the randomness enters the graph changed.

**Gradients.** Coordinate-wise, $z_j=\mu_j+\sigma_j\varepsilon_j$, so $\partial z_j/\partial\mu_j=1$ and $\partial z_j/\partial\sigma_j=\varepsilon_j$. Autograd now carries the reconstruction gradient back through $z$ into $\mu$ and $\sigma$, and from there into the encoder weights. With the log-variance head, one more Calculus AB chain-rule step through $\sigma=\exp(\frac12\ell)$ gives $\partial z_j/\partial\ell_j$; deriving it and explaining why direct sampling has no such path is p16(b).

**Reparameterized-gradient example.** Take $\mu=(0.5,-1)$, `logvar` $=(0,\log0.25)$ so $\sigma=(1,0.5)$, and $\varepsilon=(1.2,-0.4)$. Then $z=(0.5+1.2,\;-1-0.2)=(1.7,-1.2)$. For the scalar $L=z_1+z_2$, autograd reports $\partial L/\partial\mu=(1,1)$ and $\partial L/\partial\texttt{logvar}=(0.6,-0.1)$, and the cell below prints exactly these numbers next to the `.sample()` version, which has no gradient at all.

**Engineering contract.** In this unit, `reparameterize(mu, logvar, eps)` takes `eps` as an **argument** instead of drawing it inside. That keeps the function deterministic and testable, and lets the training loop draw `eps` from its own seeded `torch.Generator`. (PyTorch's `Normal(mu, std).rsample()` implements the same trick internally; this unit writes it explicitly.)

**Checkpoint 6A.** With $\mu=(2)$, `logvar` $=(\log 4)$, and $\varepsilon=(-0.5)$, compute $z$ and $\partial z/\partial\mu$.

**Checkpoint 6B.** Why does drawing $\varepsilon$ from a parameter-free distribution matter for the gradient?

**Collected answers.** $\sigma=2$, $z=2+2(-0.5)=1$, and $\partial z/\partial\mu=1$. Because then $\varepsilon$ is a constant input to the graph, and all dependence of $z$ on the parameters flows through the differentiable expression $\mu+\sigma\varepsilon$.

In [ ]:
import math
import torch

mu = torch.tensor([0.5, -1.0], requires_grad=True)
logvar = torch.tensor([0.0, math.log(0.25)], requires_grad=True)
eps = torch.tensor([1.2, -0.4])
z = mu + torch.exp(0.5 * logvar) * eps                 # reparameterized latent
z.sum().backward()
print("z =", z.tolist(), " dL/dmu =", mu.grad.tolist(), " dL/dlogvar =", logvar.grad.tolist())

mu_blocked = torch.tensor([0.5, -1.0], requires_grad=True)
z_blocked = torch.distributions.Normal(mu_blocked, torch.ones(2)).sample()
print("sample(): requires_grad =", z_blocked.requires_grad, " grad_fn =", z_blocked.grad_fn)
z_rsample = torch.distributions.Normal(mu_blocked, torch.ones(2)).rsample()
print("rsample(): requires_grad =", z_rsample.requires_grad)

## 7. Worked laboratory: per-example negative ELBO and a gradient-flow check

**The training loss.** Combining Sections 2–6 with one reparameterized sample per example, the per-example **negative ELBO** (constant $\frac d2\log2\pi$ dropped) is
$$\mathcal L_i=\underbrace{\tfrac12\sum_{j=1}^d(x_{ij}-\hat x_{ij})^2}_{\text{recon}_i:\ \text{sum over features}}+\underbrace{\tfrac12\sum_{l=1}^k\big(\mu_{il}^2+e^{\ell_{il}}-\ell_{il}-1\big)}_{\mathrm{KL}_i:\ \text{sum over latents}},$$
and the batch loss is the **mean over the batch**, $\frac1B\sum_i\mathcal L_i$. Minimizing it maximizes the average ELBO. Shapes: `x, x_hat (B,d)`; `mu, logvar (B,k)`; `recon, kl (B,)`; loss `()`.

**Hand computation** for $B=2$, $d=2$, $k=2$ with
`x = [[1,2],[0,-1]]`, `x_hat = [[0.5,2],[0,0]]`, `mu = [[0,0],[1,0]]`, `logvar = [[0,0],[0,log 2]]`:

* example 1: $\text{recon}=\frac12(0.25+0)=0.125$; $\mathrm{KL}=\frac12[(0+1-0-1)+(0+1-0-1)]=0$; total $0.125$;
* example 2: $\text{recon}=\frac12(0+1)=0.5$; $\mathrm{KL}=\frac12(1+1-0-1)+\frac12(0+2-\log2-1)=0.5+0.153426=0.653426$; total $1.153426$;
* batch loss: $(0.125+1.153426)/2=0.639213$.

If the reconstruction were averaged over features instead, example 1's recon would be $0.0625$, example 2's would be $0.25$, and the batch loss would become $0.482963$: a different objective, not a rescaling of the same one, because the KL was not divided too.

**Gradient-flow check.** The second cell builds a tiny encoder head and decoder, computes the loss with `reparameterize`, calls `backward()`, and asserts that the encoder's mean-head and log-variance-head weights received nonzero gradients. Replacing the reparameterized line with `.sample()` makes the mean-head gradient come only from the KL; the cell shows the reconstruction-only gradient is then `None`.

**Checkpoint 7A.** Recompute example 2's KL if its `logvar` were $(0,0)$.

**Checkpoint 7B.** What are the shapes of `recon`, `kl`, and the returned loss for $B=16$, $d=8$, $k=2$?

**Collected answers.** $\frac12(1+1-0-1)+\frac12(0+1-0-1)=0.5$. `recon` is `(16,)`, `kl` is `(16,)`, and the loss is a 0-dimensional tensor `()`.

In [ ]:
import math
import torch

x = torch.tensor([[1.0, 2.0], [0.0, -1.0]])
x_hat = torch.tensor([[0.5, 2.0], [0.0, 0.0]])
mu = torch.tensor([[0.0, 0.0], [1.0, 0.0]])
logvar = torch.tensor([[0.0, 0.0], [0.0, math.log(2.0)]])
recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)                          # (B,)
kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0).sum(dim=1)        # (B,)
loss = (recon + kl).mean()                                           # ()
print("recon", recon.tolist(), "kl", [round(v, 6) for v in kl.tolist()], "loss", round(loss.item(), 6))
assert torch.allclose(loss, torch.tensor(0.639213), atol=1e-6, rtol=1e-6)
wrong = (0.5 * ((x - x_hat) ** 2).mean(dim=1) + kl).mean()
print("mean-over-features variant:", round(wrong.item(), 6))

In [ ]:
import torch
from torch import nn

torch.manual_seed(20260927)
x = torch.tensor([[1.0, 2.0, 0.5], [0.0, -1.0, 1.5]])
hidden, mu_head, logvar_head = nn.Linear(3, 4), nn.Linear(4, 2), nn.Linear(4, 2)
decoder = nn.Linear(2, 3)
eps = torch.randn(2, 2, generator=torch.Generator().manual_seed(20260927))

h = torch.relu(hidden(x))
mu, logvar = mu_head(h), logvar_head(h)
z = mu + torch.exp(0.5 * logvar) * eps                                 # reparameterized
recon = 0.5 * ((x - decoder(z)) ** 2).sum(dim=1)
recon.mean().backward()                                                # reconstruction term only
print("mu-head grad norm:", mu_head.weight.grad.norm().item(), " logvar-head grad norm:", logvar_head.weight.grad.norm().item())
assert mu_head.weight.grad.abs().sum() > 0 and logvar_head.weight.grad.abs().sum() > 0

for module in (hidden, mu_head, logvar_head, decoder):
    module.zero_grad(set_to_none=True)
h = torch.relu(hidden(x))
z_blocked = torch.distributions.Normal(mu_head(h), torch.exp(0.5 * logvar_head(h))).sample()
(0.5 * ((x - decoder(z_blocked)) ** 2).sum(dim=1)).mean().backward()
print("after .sample(): mu-head grad =", mu_head.weight.grad)

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Sampling inside the graph with `.sample()`, `.detach()`, or `.item()`.* The encoder gets no reconstruction gradient. Fix: `z = mu + torch.exp(0.5 * logvar) * eps` with `eps` drawn independently.
2. *Using `exp(logvar)` as the standard deviation.* That is the variance; it squares the noise scale. Fix: `torch.exp(0.5 * logvar)`.
3. *Sign errors.* The loss to **minimize** is recon **plus** KL (the negative ELBO). Minimizing recon minus KL rewards an arbitrarily large KL.
4. *Reduction mismatch.* Mean over features (or over latents) instead of the sums the derivation produces. Fix: sum over features and latents per example, then mean over the batch.
5. *Drawing `eps` inside `reparameterize`.* Makes the function untestable and couples it to global RNG state. Fix: pass `eps` in; draw it from a dedicated seeded generator in the loop.
6. *Claiming the ELBO equals $\log p(x)$.* It is a lower bound; the gap is $\mathrm{KL}(q\|p(z\mid x))\ge0$.

**Exam connections.** p05 asks which quantity reparameterization makes differentiable; p11 implements `reparameterize(mu, logvar, eps)` with a gradient probe; p12 implements `negative_elbo(x, x_hat, mu, logvar)` with the reduction above; p16 derives the ELBO identity and the gradient path; p21 compares an autoencoder with a VAE under constraints; p23 and p24 (Session 5) repair a broken loss and audit a non-differentiable sampling step.

**Checkpoint 8A.** A loss is written `recon.mean() - kl.mean()`. What happens to the encoder's variances during training?

**Checkpoint 8B.** Which practice derives $\partial z/\partial\,\texttt{logvar}$?

**Collected answers.** Minimizing `-kl` rewards a large KL, so the encoder pushes means and log-variances to extremes (KL grows without bound) instead of matching the prior. p16, part (b).

## 9. Forward-only deeper topic: tighter bounds and diffusion as chained reparameterizations

Forward-only deeper topic.
Averaging several importance-weighted samples inside the logarithm gives bounds tighter than the ELBO (importance-weighted autoencoders), and the **score-function** (REINFORCE) estimator gives gradients through sampling when no reparameterization exists, for example with discrete latents, at the cost of high variance.
In `B2-023-generative-models-diffusion`, a diffusion model's forward process is a chain of reparameterized Gaussian steps, $x_t=\sqrt{\alpha_t}\,x_{t-1}+\sqrt{1-\alpha_t}\,\varepsilon_t$, and its training objective is again an ELBO whose terms are Gaussian KLs.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-022.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is the score-function estimator used in any B2-022 practice?

**Checkpoint 9B.** In the diffusion step above, which symbol plays the role of $\varepsilon$ in $z=\mu+\sigma\varepsilon$?

**Collected answers.** No; it is forward-only and not assessed. $\varepsilon_t$, scaled by $\sqrt{1-\alpha_t}$ (the standard deviation), with mean $\sqrt{\alpha_t}\,x_{t-1}$.